# SfM Pipeline Implementation IMC25
- Implementation of sparse reconstruction pipeline using COLMAP

In [63]:
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import os
import json
import struct
from tqdm.auto import tqdm

## Notes 

- This notebook requires a COLMAP installation: Requires COLMAP >= 3.7 installed system-wide
- Implementation of the full SfM pipeline that outputs the submission.csv for Kaggle and sparse 3D point cloud for 3D reconstruction 
- Will add more documentation later


In [64]:
#check that colmap works (requirement for notebook)
!colmap feature_extractor -h

COLMAP 3.7 (Commit Unknown on Unknown without CUDA)

Options can either be specified via command-line or by defining
them in a .ini project file passed to `--project_path`.

  -h [ --help ] 
  --random_seed arg (=0)
  --log_to_stderr arg (=0)
  --log_level arg (=2)
  --project_path arg
  --database_path arg
  --image_path arg
  --camera_mode arg (=-1)
  --image_list_path arg
  --descriptor_normalization arg (=l1_root)
                                        {'l1_root', 'l2'}
  --ImageReader.mask_path arg
  --ImageReader.camera_model arg (=SIMPLE_RADIAL)
  --ImageReader.single_camera arg (=0)
  --ImageReader.single_camera_per_folder arg (=0)
  --ImageReader.single_camera_per_image arg (=0)
  --ImageReader.existing_camera_id arg (=-1)
  --ImageReader.camera_params arg
  --ImageReader.default_focal_length_factor arg (=1.2)
  --ImageReader.camera_mask_path arg
  --SiftExtraction.num_threads arg (=-1)
  --SiftExtraction.use_gpu arg (=1)
  --SiftExtraction.gpu_index arg (=-1)
  --SiftExtract

In [65]:
#Forcing COLMAP to run headless

os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.environ["DISPLAY"] = ""
os.environ["XDG_RUNTIME_DIR"] = "/tmp"

ENV = os.environ

## Setup
- Data loading
- COLMAP setup

In [66]:
#Sfm pipeline steps for status bar updates
SCENE_STEPS = [
    "prepare_images",
    "feature_extraction",
    "matching",
    "mapping",
    "pose_extraction",
]


#Defining paths
DATA_ROOT = Path("./data/train")
CACHE_ROOT = Path("./cache")
SFM_ROOT = CACHE_ROOT / "sfm"
SFM_ROOT.mkdir(exist_ok=True)

CLUSTERS_CSV = CACHE_ROOT / "clusters.csv"
SUBMISSION_CSV = Path("submission.csv")

# COLMAP settings
COLMAP_BIN = "colmap"
CAMERA_MODEL = "SIMPLE_RADIAL"
MAX_IMAGE_SIZE = 2000   #this can be adjusted based on how fast the pipeline can process images

# Load clusters
clusters_df = pd.read_csv(CLUSTERS_CSV)

# Ignore outliers
clusters_df = clusters_df[clusters_df.scene != "outliers"]

In [67]:
def run(cmd, cwd=None, timeout=900):
    print(" ".join(map(str, cmd)))
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError("COLMAP command failed")


## SfM Pipeline: Scene Reconstructions

In [68]:
#Qutaernation: rotation matrix

def qvec2rotmat(qvec):
    w, x, y, z = qvec
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*z*w,     2*x*z + 2*y*w],
        [2*x*y + 2*z*w,     1 - 2*x*x - 2*z*z, 2*y*z - 2*x*w],
        [2*x*z - 2*y*w,     2*y*z + 2*x*w,     1 - 2*x*x - 2*y*y]
    ])

In [69]:
def reconstruct_scene(dataset, scene, image_ids, pbar=None):
    scene_dir = SFM_ROOT / dataset / scene
    img_dir = scene_dir / "images"
    db_path = scene_dir / "database.db"
    sparse_dir = scene_dir / "sparse"

    #resetting if scene directory exists
    if scene_dir.exists():
        shutil.rmtree(scene_dir)

    scene_dir.mkdir(parents=True)
    img_dir.mkdir()

    if pbar:
        pbar.update(1)  # updates prepare_images step


    # Copy images
    for img_id in image_ids:
        matches = list((DATA_ROOT / dataset).rglob(f"{img_id}.*"))
        if not matches:
            raise RuntimeError(f"Image not found: {img_id}")
        shutil.copy(matches[0], img_dir / matches[0].name)

    if len(list(img_dir.iterdir())) == 0:
        raise RuntimeError("No images copied")

    # Feature extraction
    run([
        COLMAP_BIN, "feature_extractor",
        "--database_path", db_path,
        "--image_path", img_dir,
        "--ImageReader.camera_model", CAMERA_MODEL,
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", "0",
        "--SiftExtraction.max_image_size", str(MAX_IMAGE_SIZE),
        "--SiftExtraction.num_threads", "4",
        "--SiftExtraction.max_num_features", "4096", #reduced this for better processing speed
        "--SiftExtraction.first_octave", "0",
    ])
    if pbar: pbar.update(1)

    # Matching
    run([
        COLMAP_BIN, "exhaustive_matcher",
        "--database_path", db_path,
        "--SiftMatching.use_gpu", "0",
        "--SiftMatching.num_threads", "4",
    ])
    if pbar: pbar.update(1)

    #ensuring clean sparse dir
    if sparse_dir.exists() and not sparse_dir.is_dir():
      sparse_dir.unlink()         
    if sparse_dir.exists():
      shutil.rmtree(sparse_dir)   
    sparse_dir.mkdir(parents=True, exist_ok=True)

    print("Images in scene:", len(list(img_dir.iterdir())))
    print("Image path exists:", img_dir.exists())


    # Mapping
    run([
        COLMAP_BIN, "mapper",
        "--database_path", db_path,
        "--image_path", img_dir,

        "--output_path", sparse_dir,
        "--Mapper.min_num_matches", "8",
        "--Mapper.init_num_trials", "50",
        "--Mapper.max_reg_trials", "2",
        "--Mapper.ba_global_max_num_iterations", "5",
        "--Mapper.ba_global_max_refinements", "1",
    ], timeout=3600) #with increased timeout

    if pbar: pbar.update(1)

    model_dir = sparse_dir / "0"
    if not model_dir.exists():
        return None

    return model_dir

## SfM per Scene + Pose extraction


In [70]:
# converting bin files containing camera poses to txt files for stability

def extract_poses(model_dir, scene_dir, dataset, scene):
    txt_dir = scene_dir / "sparse_txt"
    if txt_dir.exists():
        shutil.rmtree(txt_dir)
    txt_dir.mkdir()

    run([
        COLMAP_BIN, "model_converter",
        "--input_path", model_dir,
        "--output_path", txt_dir,
        "--output_type", "TXT",
    ])

    poses = []
    with open(txt_dir / "images.txt") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if line.startswith("#") or not line:
            i += 1
            continue

        parts = line.split()
        if len(parts) < 10:
            i += 1
            continue

        # Extract image pose
        _, qw, qx, qy, qz, tx, ty, tz, _, name = parts[:10]

        qvec = np.array([float(qw), float(qx), float(qy), float(qz)])
        tvec = np.array([float(tx), float(ty), float(tz)])

        R = qvec2rotmat(qvec)
        C = -R.T @ tvec

        poses.append({
            "image_id": Path(name).stem,
            "dataset": dataset,
            "scene": scene,
            "image": name,
            "rotation_matrix": ";".join(map(str, R.T.flatten())),
            "translation_vector": ";".join(map(str, C)),
        })

        i += 2

    return pd.DataFrame(poses)

In [71]:
#Save per scene outputs
def save_scene_outputs(scene_dir, model_dir, dataset, scene, image_ids):
    poses_df = extract_poses(model_dir, scene_dir, dataset, scene)
    poses_df.to_csv(scene_dir / "poses.csv", index=False)

    stats = {
        "dataset": dataset,
        "scene": scene,
        "num_images_total": len(image_ids),
        "num_images_registered": len(poses_df),
        "success": len(poses_df) >= 2,
    }

    with open(scene_dir / "stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    return poses_df


# Run SfM Pipeline per Scene

In [72]:
clusters_df = pd.read_csv(CLUSTERS_CSV)
clusters_df = clusters_df[clusters_df.scene != "outliers"]

groups = list(clusters_df.groupby(["dataset", "scene"]))
all_rows = []

for i, ((dataset, scene), g) in enumerate(groups, 1):
    image_ids = g.image_id.tolist()

    print(f"\n[{i}/{len(groups)}] {dataset} / {scene}")

    if len(image_ids) < 2:
        print("Skipping: too few images")
        continue

    with tqdm(total=len(SCENE_STEPS), desc=f"{dataset}/{scene}", leave=False) as pbar:
        try:
            model_dir = reconstruct_scene(dataset, scene, image_ids, pbar)
            if model_dir is None:
                print("Reconstruction failed")
                continue

            scene_dir = SFM_ROOT / dataset / scene
            poses_df = save_scene_outputs(scene_dir, model_dir, dataset, scene, image_ids)
            pbar.update(1)

            all_rows.append(poses_df)
            print(f"Finished {dataset}/{scene}")

        except Exception as e:
            print(f"Skipping {dataset}/{scene} due to error:")
            print(e)



[1/151] ETs / cluster_0001


ETs/cluster_0001:   0%|          | 0/5 [00:00<?, ?it/s]

Skipping ETs/cluster_0001 due to error:
Image not found: another_et_another_et001

[2/151] ETs / cluster_0002


ETs/cluster_0002:   0%|          | 0/5 [00:00<?, ?it/s]

Skipping ETs/cluster_0002 due to error:
Image not found: et_et000

[3/151] ETs / cluster_0003


...

Skipping pt_stpeters_stpauls/cluster_0008 due to error:
Image not found: st_pauls_cathedral_06850079_5924336712

[150/151] pt_stpeters_stpauls / cluster_0009


pt_stpeters_stpauls/cluster_0009:   0%|          | 0/5 [00:00<?, ?it/s]

Skipping pt_stpeters_stpauls/cluster_0009 due to error:
Image not found: st_pauls_cathedral_04223687_109190093

[151/151] stairs / cluster_0001


stairs/cluster_0001:   0%|          | 0/5 [00:00<?, ?it/s]

Skipping stairs/cluster_0001 due to error:
Image not found: stairs_split_2_1710453740954


## Output

In [73]:
submission_df = pd.concat(all_rows, ignore_index=True)
submission_df = submission_df[
    ["image_id", "dataset", "scene", "image", "rotation_matrix", "translation_vector"]
]
submission_df.to_csv(SUBMISSION_CSV, index=False)

print("Saved submission.csv")
display(submission_df.head())


ValueError: No objects to concatenate